# Final Hybrid RAG (Graph + Vector)
This notebook combines Neo4j Graph traversal with OpenAI Vector Search.
It handles separate keys for LLM and Embeddings correctly.

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

True

In [2]:
# 1. Initialize Connections
try:
    graph = Neo4jGraph(
        url=os.getenv('NEO4J_URI'),
        username=os.getenv('NEO4J_USERNAME'),
        password=os.getenv('NEO4J_PASSWORD')
    )
    print("✅ Neo4j Connected")
except Exception as e:
    print(f"❌ Neo4j Error: {e}")

# LLM (GPT-4o)
llm = ChatOpenAI(
    api_key=os.getenv('OpenAi_api'),
    base_url='https://aigateway.ntictsolution.com/v1',
    model='gpt-4o',
    temperature=0
)

# Embeddings (Large) - Uses specialized key if available
embed_key = os.getenv('OpenAi_api_embbeding')
if not embed_key:
    print("⚠️ Warning: 'OpenAi_api_embbeding' not found. Fallback to 'OpenAi_api' (This may fail if permissions are restricted).")
    embed_key = os.getenv('OpenAi_api')

embeddings = OpenAIEmbeddings(
    model='text-embedding-3-large',
    openai_api_base='https://aigateway.ntictsolution.com/v1',
    openai_api_key=embed_key
)

✅ Neo4j Connected


In [ ]:
# Improved Hybrid RAG Functions

def create_fulltext_index(graph):
    """Creates a fulltext index on Entity nodes for better keyword search."""
    try:
        graph.query("""
            CREATE FULLTEXT INDEX entity_id_index IF NOT EXISTS 
            FOR (n:Entity) ON EACH [n.id]
        """)
        print("✅ Fulltext Index 'entity_id_index' checks out.")
    except Exception as e:
        print(f"⚠️ Could not create fulltext index (might already exist): {e}")

def extract_entities(llm, question):
    """Uses LLM to find the most important entities/keywords."""
    prompt = f"""
    Extract the 2-3 most important entities or keywords from this question for a database search.
    Return ONLY a comma-separated list of terms. NO explanation.

    Question: {question}
    """
    response = llm.invoke(prompt).content
    terms = [t.strip() for t in response.split(',') if t.strip()]
    print(f"🧩 Extracted search terms: {terms}")
    return terms

def retrieve_graph_context(graph, llm, question, limit=15):
    """Smartly gets relevant entities using Fulltext Search + LLM extraction."""
    create_fulltext_index(graph)
    search_terms = extract_entities(llm, question)
    if not search_terms:
        return ""

    lucene_query = " OR ".join([f"{term}~" for term in search_terms])
    cypher_query = """
    CALL db.index.fulltext.queryNodes("entity_id_index", $query, {limit: 5})
    YIELD node, score
    MATCH (node)-[r]-(connected)
    RETURN node, type(r) as relation, connected, score
    LIMIT $limit
    """
    
    try:
        results = graph.query(cypher_query, {"query": lucene_query, "limit": limit})
    except Exception as e:
        print(f"⚠️ Fulltext search failed, fallback to basic CONTAINS: {e}")
        return ""

    context_lines = []
    for row in results:
        s_id = row['node'].get('id', 'Unknown')
        t_id = row['connected'].get('id', row['connected'].get('text', '')[:50] + "...")
        context_lines.append(f"{s_id} --[{row['relation']}]--> {t_id}")
    return "\n".join(set(context_lines))

def vector_retrieve(graph, embeddings, question, k=10):
    print('🔍 Searching Vector Index...')
    try:
        q_embedding = embeddings.embed_query(question)
        return graph.query("""
            CALL db.index.vector.queryNodes('doc_embedding', $k, $emb)
            YIELD node, score
            RETURN node.text as text, score
        """, {'emb': q_embedding, 'k': k})
    except Exception as e:
        print(f"⚠️ Vector Search Failed: {e}")
        return []

def hybrid_query(question):
    # 1. Get Context
    graph_text = retrieve_graph_context(graph, llm, question)
    vector_res = vector_retrieve(graph, embeddings, question)
    
    # 2. Combine
    context = '### GRAPH SOURCES:\n'
    context += graph_text if graph_text else "No graph connections found.\n"
    
    context += '\n### TEXT SOURCES:\n'
    if vector_res:
        for row in vector_res:
            context += f"- {row['text']}\n"
    else:
        context += "No text found.\n"
        
    print(f"\n📋 Context Length: {len(context)} chars")

    print(f"\n📋 Context : {context}")
    # 3. Ask LLM
    prompt = f"""
    You are a Hybrid RAG Assistant. Answer ONLY using the context below.
    If the answer is not present, say "I don't know".
    
    Context:
    {context}
    
    Question: {question}
    
    Answer:
    """
    return llm.invoke(prompt).content


In [5]:
# Test it!
response = hybrid_query('what team is max drive for?')
print("\n🤖 FINAL ANSWER:")
print(response)

✅ Fulltext Index 'entity_id_index' checks out.
🧩 Extracted search terms: ['max', 'drive', 'team']
🔍 Searching Vector Index...

📋 Context Length: 8097 chars

🤖 FINAL ANSWER:
I don't know.
